In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# Why saline is 0.9 % (Illustration 11.5-4)

This notebook works **Illustration 11.5-4**: the osmotic pressure of a solution that is
99.4 mol % water. The answer is above 7 bar, which is the surprise the illustration is
built around, and it is the reason an intravenous fluid has to be matched to blood
rather than merely made from clean water.

**Three calculations, not one.** Equation 11.5-4 is
$\Pi = -(RT/\underline{V})\ln(x_W\gamma_W)$, so the answer is a logarithm of a number
within one percent of unity multiplied by 1377 bar. Three things therefore have to be
right, and the notebook does them in order:

1. the **counting** -- sodium chloride ionizes completely, so 0.1554 mol of salt puts
   0.3108 mol of solute into 55.51 mol of water;
2. the **ideal estimate** -- $\gamma_W = 1$, which the illustration works first;
3. the **activity coefficient of the water**, which differs from one in the fourth
   decimal place and moves the answer by several percent. That is the illustration's
   own closing point: *"in the case of the osmotic pressure, the activity coefficient
   correction should not be neglected."*

**And step 3 does not reproduce the printed number.** The equation Appendix A9.3 prints
and the equation Appendix A9.3 derives differ by a factor of three in their correction
term, and the notebook settles which is which by integrating the Gibbs-Duhem equation
numerically over the book's own measured activity coefficients. That is done below
rather than asserted.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:
import sys; sys.path.insert(0, "..")
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.integrate import quad

from thermo.electrolytes import DebyeHuckel, ILLUSTRATION_9_10_2_NACL, MW_WATER
from thermo.osmotic import osmotic_pressure, electrolyte_mole_fraction

T = 298.15
V_WATER = 18e-6          # m^3/mol, the value the illustration uses
MW_NACL = 58.44

# 0.9 wt % NaCl: 9 g of salt in 1000 g of solution, so 991 g of water.
g_per_kg_water = 9.0 / 991.0 * 1000.0
molality = g_per_kg_water / MW_NACL
x_water = electrolyte_mole_fraction(molality, nu=2)

print("  The counting, and where the factor of two enters")
print(f"    9 g NaCl per 991 g water = {g_per_kg_water:.4f} g/kg"
      f"        SIS 9.0817")
print(f"    molality = {molality:.4f} mol/kg"
      f"                        SIS 0.1554")
print(f"    x_water  = 55.51/(55.51 + 2 x {molality:.4f}) = {x_water:.6f}"
      f"   SIS 0.994 43")
print(f"    if the salt did NOT ionize, x_water would be"
      f" {electrolyte_mole_fraction(molality, nu=1):.6f}")

Pi_ideal = osmotic_pressure(x_water, V_WATER, T) / 1e5
print(f"\n  RT/V for water at 25 C = {8.314 * T / V_WATER / 1e5:.1f} bar")
print(f"  ideal-solution estimate, gamma_W = 1: Pi = {Pi_ideal:.3f} bar"
      f"     SIS 7.694")
print(f"  and if the salt did not ionize:"
      f" {osmotic_pressure(electrolyte_mole_fraction(molality, 1), V_WATER, T)/1e5:.3f} bar")

  The counting, and where the factor of two enters
    9 g NaCl per 991 g water = 9.0817 g/kg        SIS 9.0817
    molality = 0.1554 mol/kg                        SIS 0.1554
    x_water  = 55.51/(55.51 + 2 x 0.1554) = 0.994432   SIS 0.994 43
    if the salt did NOT ionize, x_water would be 0.997208

  RT/V for water at 25 C = 1377.1 bar
  ideal-solution estimate, gamma_W = 1: Pi = 7.690 bar     SIS 7.694
  and if the salt did not ionize: 3.850 bar



## The activity coefficient of the water

Appendix A9.3 integrates the Gibbs-Duhem equation to get the solvent's activity
coefficient from the salt's. For sodium chloride at 25 °C the appendix uses
Eq. 9.10-18 for the salt,

$$\ln\gamma_\pm = -\frac{\alpha|z_+z_-|\sqrt{I}}{1+\sqrt{I}} + 0.137\,I$$

and reports $\gamma_{\rm H_2O} = 1.000\,13$ at this concentration, giving
$x_W\gamma_W = 0.994\,57$ and $\Pi = 7.498$ bar.

`DebyeHuckel.ln_solvent_activity` computes that, and it computes it two ways, because
the appendix as printed and the appendix as derived are not the same equation. Both
appear in the printed pages:

- the appendix's stated limit $\sigma(0) = 1/3$, which makes the correction term
  $\alpha|z_+z_-|\sqrt{I}\,\sigma$;
- the appendix's printed prefactor $\alpha/3$, which divides that same term by three
  again.

They cannot both be right, and they differ by 6 % in $\ln(x_W\gamma_W)$ -- which is 6 %
of the osmotic pressure.

In [3]:

model = DebyeHuckel("NaCl", T=T, beta_a=1.0, delta=0.137)
print(f"  {model!r}")
print(f"  alpha at 25 C from Table 9.10-1: {model.alpha}")

for as_printed, label in ((True, "as Appendix A9.3 prints it"),
                          (False, "as Appendix A9.3 derives it")):
    lnxg = float(model.ln_solvent_activity(molality, as_printed=as_printed))
    gamma = float(model.solvent_gamma(molality, as_printed=as_printed))
    Pi = osmotic_pressure(x_water, V_WATER, T, gamma_solvent=gamma) / 1e5
    print(f"\n  {label}")
    print(f"    x_W gamma_W = {np.exp(lnxg):.6f}      SIS 0.994 57")
    print(f"    gamma_W     = {gamma:.6f}      SIS 1.000 13")
    print(f"    Pi          = {Pi:.3f} bar        SIS 7.498")

  DebyeHuckel('NaCl', T=298.15, beta_a=1, delta=0.137)  # Eq. 9.10-18
  alpha at 25 C from Table 9.10-1: 1.175

  as Appendix A9.3 prints it
    x_W gamma_W = 0.994534      SIS 0.994 57
    gamma_W     = 1.000103      SIS 1.000 13
    Pi          = 7.548 bar        SIS 7.498

  as Appendix A9.3 derives it
    x_W gamma_W = 0.994879      SIS 0.994 57
    gamma_W     = 1.000450      SIS 1.000 13
    Pi          = 7.070 bar        SIS 7.498



The printed form reproduces the illustration's $\gamma_{\rm H_2O} = 1.000\,13$ to four
decimal places, so that is the equation the printed number came from. Which does not
make it the right one. The test that settles it uses no data at all beyond the book's
own: take Eq. 9.10-18 for the salt, integrate

$$\ln(x_S\gamma_S) = -m_S\,\nu\left[M + \int_0^M M\,{\rm d}\ln\gamma_\pm\right]$$

numerically, and compare with each closed form.

In [4]:

def gibbs_duhem_ln_activity(ln_gamma_pm, M_max, nu=2, mw=MW_WATER):
    """ln(x_S gamma_S) by numerical integration of the Gibbs-Duhem equation.

    `ln_gamma_pm` is any callable of molality. The integrand M d(ln gamma_pm)/dM is
    evaluated by central differences, so this knows nothing about the closed form it
    is being used to check.
    """
    def integrand(M):
        h = 1e-7 * max(M, 1e-3)
        return M * (ln_gamma_pm(M + h) - ln_gamma_pm(M - h)) / (2 * h)
    integral, _ = quad(integrand, 1e-12, M_max, limit=400)
    return -mw * nu * (M_max + integral)


print("  ln(x_W gamma_W) three ways")
print(f"  {'M':>7s} {'numerical Gibbs-Duhem':>22s} {'derived':>12s} {'as printed':>12s}")
for M in (molality, 1.0, 6.0):
    num = gibbs_duhem_ln_activity(lambda m: float(model.ln_gamma_pm(m)), M)
    der = float(model.ln_solvent_activity(M))
    pr = float(model.ln_solvent_activity(M, as_printed=True))
    print(f"  {M:7.4f} {num:22.8f} {der:12.8f} {pr:12.8f}")

print("\n  The derived form is the integral of the book's own Eq. 9.10-18, to eight")
print("  decimal places. The printed form is not the integral of anything here.")

  ln(x_W gamma_W) three ways
        M  numerical Gibbs-Duhem      derived   as printed
   0.1554            -0.00513367  -0.00513367  -0.00548059
   1.0000            -0.03365625  -0.03365625  -0.03686275
   6.0000            -0.27587921  -0.27587921  -0.29514374

  The derived form is the integral of the book's own Eq. 9.10-18, to eight
  decimal places. The printed form is not the integral of anything here.



That check is circular in one respect: it shows the closed form integrates Eq. 9.10-18
correctly, not that Eq. 9.10-18 describes sodium chloride. So do it again with the
*measured* activity coefficients -- Illustration 9.10-2's own table, ten values of
$\gamma_\pm$ for aqueous NaCl from 0.1 to 6 molal, which the package already carries.
Below 0.1 molal, where the table stops, the Debye-Hückel limiting law supplies the
slope, and it is exact in that limit by construction.

In [5]:

M_meas, g_meas = ILLUSTRATION_9_10_2_NACL[:, 0], ILLUSTRATION_9_10_2_NACL[:, 1]

# ln gamma_pm is nearly linear in sqrt(I), so spline it there. The end condition at
# I = 0 is the limiting law's slope, -alpha|z+z-|, which is not a fitted number.
spline = CubicSpline(np.concatenate([[0.0], np.sqrt(M_meas)]),
                     np.concatenate([[0.0], np.log(g_meas)]),
                     bc_type=((1, -model.slope), "not-a-knot"))

def ln_gamma_measured(M):
    return float(spline(np.sqrt(M)))

print("  Water activity from the MEASURED gamma_pm of Illustration 9.10-2")
print(f"  {'M':>7s} {'measured route':>15s} {'derived':>10s} {'as printed':>12s}"
      f" {'derived err':>12s} {'printed err':>12s}")
for M in (molality, 1.0, 2.0, 3.0, 4.0, 6.0):
    a_meas = np.exp(gibbs_duhem_ln_activity(ln_gamma_measured, M))
    a_der = float(np.exp(model.ln_solvent_activity(M)))
    a_pr = float(np.exp(model.ln_solvent_activity(M, as_printed=True)))
    print(f"  {M:7.4f} {a_meas:15.6f} {a_der:10.6f} {a_pr:12.6f}"
          f" {100*(a_der/a_meas - 1):11.2f} % {100*(a_pr/a_meas - 1):11.2f} %")

print(f"\n  rms of Eq. 9.10-18 against those same measured gamma_pm:"
      f" {model.rms(M_meas, g_meas):.4f} in ln gamma_pm")

  Water activity from the MEASURED gamma_pm of Illustration 9.10-2
        M  measured route    derived   as printed  derived err  printed err
   0.1554        0.994826   0.994879     0.994534        0.01 %       -0.03 %
   1.0000        0.966871   0.966904     0.963808        0.00 %       -0.32 %
   2.0000        0.931561   0.930691     0.924485       -0.09 %       -0.76 %
   3.0000        0.893290   0.891243     0.882342       -0.23 %       -1.23 %
   4.0000        0.851826   0.849083     0.837916       -0.32 %       -1.63 %
   6.0000        0.759887   0.758905     0.744425       -0.13 %       -2.03 %

  rms of Eq. 9.10-18 against those same measured gamma_pm: 0.0219 in ln gamma_pm


The derived form tracks the measured water activity to better than 0.35 % over the whole
range, and to 0.01 % at the concentration this illustration is about. The printed form
is low at every one of the six concentrations, by 0.03 % at 0.155 molal and 2 % at
6 molal. A one-sided error that grows with the size of the correction term is what a
missing factor looks like; scatter would change sign.

**This is a Chapter 9 correction, found in Chapter 11.** Appendix A9.3's two
equations require $\alpha$ where they print $\alpha/3$ -- or equivalently require
$\sigma(0)=1$ where the text says $1/3$ -- and Illustration A9.3-1's table of
$\gamma_{\rm H_2O}$ follows the printed form. Illustration 11.5-4's
$\gamma_{\rm H_2O} = 1.000\,13$ and $\Pi = 7.498$ bar follow it too.

## So what is the osmotic pressure of blood?

In [6]:

gamma_meas = float(np.exp(gibbs_duhem_ln_activity(ln_gamma_measured, molality))) / x_water
rows = [
    ("no ionization, ideal", osmotic_pressure(
        electrolyte_mole_fraction(molality, 1), V_WATER, T) / 1e5),
    ("complete ionization, ideal (SIS 7.694)", Pi_ideal),
    ("gamma_W from A9.3 as printed (SIS 7.498)", osmotic_pressure(
        x_water, V_WATER, T,
        float(model.solvent_gamma(molality, as_printed=True))) / 1e5),
    ("gamma_W from A9.3 as derived", osmotic_pressure(
        x_water, V_WATER, T, float(model.solvent_gamma(molality))) / 1e5),
    ("gamma_W from the measured gamma_pm", osmotic_pressure(
        x_water, V_WATER, T, gamma_meas) / 1e5),
]
print(f"  {'route':42s} {'Pi (bar)':>9s}")
for name, Pi in rows:
    print(f"  {name:42s} {Pi:9.3f}")

best = rows[-1][1]
print(f"\n  The best of these is the last: {best:.2f} bar, from measured activity")
print(f"  coefficients rather than from a correlation of them.")
print(f"  The ideal estimate is {100*(Pi_ideal/best - 1):.0f} % high, and ignoring the")
print(f"  dissociation entirely would halve the answer.")

  route                                       Pi (bar)
  no ionization, ideal                           3.850
  complete ionization, ideal (SIS 7.694)         7.690
  gamma_W from A9.3 as printed (SIS 7.498)       7.548
  gamma_W from A9.3 as derived                   7.070
  gamma_W from the measured gamma_pm             7.144

  The best of these is the last: 7.14 bar, from measured activity
  coefficients rather than from a correlation of them.
  The ideal estimate is 8 % high, and ignoring the
  dissociation entirely would halve the answer.



## What the number is for

Seven bar across a cell membrane is a large pressure -- about seven atmospheres, or the
pressure at 70 meters of water. A red blood cell placed in pure water has that much
pressure driving water inward, and it bursts. That is why every fluid in clinical use --
dialysate, blood substitutes, organ preservation solutions, contact lens saline -- is
made isotonic with blood, and why the number 0.9 wt % appears on saline bags rather
than any convenient round concentration.

The thermodynamics behind it is a single logarithm. It has to be computed rather
than estimated because the answer is the logarithm of a number near one: the
dissociation is a factor of two, the departure from ideality is 8 %, and neither can be
waved away.

## Your turn

1. Problem 11.5-10 asks for the osmotic pressure of a flat Coca-Cola -- 39 g of fructose
   in 355 mL, molecular weight 180.16. Compute it, and then compare it with blood.
   Which has more dissolved solute by mass, and which has more by moles?
2. Work the same calculation for a 0.9 wt % solution of calcium chloride instead of
   sodium chloride. How much of the change is the different molecular weight and how
   much is $\nu = 3$?
3. Problem 11.5-11 is about shrimp: 70 mmol/kg of osmolytes at the surface and
   300 mmol/kg at 3 km depth, in fluids that also carry 0.9 wt % NaCl. Compute the
   osmotic pressure at both depths and compare with the hydrostatic pressure at 3 km.
   Does the osmolyte account for it?
4. The illustration takes $\underline{V}_{\rm water}$ as constant. Water is compressible
   at 4.5 × 10⁻⁵ bar⁻¹; how large is the Poynting term of Eq. 11.5-3 that was dropped in
   getting to Eq. 11.5-4, at $\Pi = 7$ bar?
5. Run `ln_solvent_activity` over 0 to 6 molal in both forms and plot $\gamma_{\rm H_2O}$
   against $\sqrt{M}$. Where does the difference between them first exceed the precision
   Illustration A9.3-1's table is printed to?